In [ ]:
# ================== 0) Mount & Imports ==================
from google.colab import drive
drive.mount('/content/drive')

import os, random, pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

logsoft = nn.LogSoftmax(dim=1)

# ================== GroupNorm ==================
GN_GROUPS = 32
def make_gn(C: int) -> nn.GroupNorm:
    g = min(GN_GROUPS, C)
    while g > 1 and (C % g) != 0:
        g //= 2
    return nn.GroupNorm(num_groups=max(1, g), num_channels=C)

# ================== 1) ResNet18 Backbone + Multi-Head (GroupNorm) ==================
def conv3x3(in_planes: int, out_planes: int, stride: int = 1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)

class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes: int, planes: int, stride: int = 1):
        super().__init__()
        self.conv1 = conv3x3(in_planes, planes, stride)
        self.gn1 = make_gn(planes)
        self.conv2 = conv3x3(planes, planes, 1)
        self.gn2 = make_gn(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, kernel_size=1,
                          stride=stride, bias=False),
                make_gn(planes * self.expansion)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        out = out + self.shortcut(x)
        out = torch.relu(out)
        return out

class ResNet18Backbone(nn.Module):
    def __init__(self, nf: int = 64):
        super().__init__()
        block = BasicBlock
        num_blocks = [2, 2, 2, 2]
        self.expansion = block.expansion
        self.nf = nf
        self.in_planes = nf

        self.conv1 = conv3x3(3, nf)
        self.gn1 = make_gn(nf)
        self.layer1 = self._make_layer(block, nf,     num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, nf * 2, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, nf * 4, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, nf * 8, num_blocks[3], stride=2)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        in_planes = self.in_planes
        for s in strides:
            layers.append(block(in_planes, planes, s))
            in_planes = planes * block.expansion
        self.in_planes = in_planes
        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = torch.relu(self.gn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = torch.nn.functional.avg_pool2d(out, out.shape[2])
        feat = out.view(out.size(0), -1)
        return feat

    @property
    def out_dim(self) -> int:
        return self.nf * 8

class MultiHeadNet(nn.Module):
    def __init__(self, backbone: ResNet18Backbone):
        super().__init__()
        self.backbone = backbone
        self.heads = nn.ModuleDict()

    def add_head(self, task_name: str, num_classes: int):
        if task_name in self.heads:
            raise ValueError(f"Head '{task_name}' exists.")
        head = nn.Linear(self.backbone.out_dim, num_classes)
        head = head.to(next(self.backbone.parameters()).device)
        self.heads[task_name] = head

    def forward(self, x: torch.Tensor, task_name: str) -> torch.Tensor:
        if task_name not in self.heads:
            raise ValueError(f"Head '{task_name}' not found.")
        feat = self.backbone(x)
        logits = self.heads[task_name](feat)
        return logits

# ================== 2) Paths / Settings ==================
BASE = "/content/drive/MyDrive/ML_Project/project_files/First_benchmark"
MODEL_PATH = f"{BASE}/Gtask3_best_test_for_finetune.pth"

SAVE_FISHER_FULL    = f"{BASE}/Gfisher_task3_cifar10_full.pkl"
SAVE_TOPK_PATH      = f"{BASE}/Gfisher_task3_cifar10_topk.pkl"
SAVE_NEIGHBORS_PATH = f"{BASE}/Gfisher_task3_cifar10_neighbors.pkl"

TOP_K = 400000
NUM_FISHER_SAMPLES = 2000

os.makedirs(BASE, exist_ok=True)

# ================== 3) CIFAR-10 + Task 3 Label Remapping (Classes 4 and 5 → {0,1}) ==================
tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.4914, 0.4822, 0.4465),
                         std=(0.2470, 0.2435, 0.2616)),
])

train_full = datasets.CIFAR10(root="./data", train=True, download=True, transform=tf)

targets_full = train_full.targets if hasattr(train_full, "targets") \
    else [train_full[i][1] for i in range(len(train_full))]
idx_task3 = [i for i, t in enumerate(targets_full) if int(t) in [4, 5]]

class RemapCIFARSubset(Dataset):
    def __init__(self, dataset, indices, keep_classes=[4,5]):
        self.dataset = dataset
        self.indices = list(indices)
        self.keep = sorted(int(c) for c in keep_classes)
        self.mapping = {c:i for i,c in enumerate(self.keep)}
        if hasattr(dataset, "targets"):
            orig_targets = [int(dataset.targets[i]) for i in self.indices]
        else:
            orig_targets = [int(dataset[i][1]) for i in range(len(dataset))]
        self.remapped = [self.mapping[y] for y in orig_targets]
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        x, _ = self.dataset[self.indices[idx]]
        return x, self.remapped[idx]

train_task3 = RemapCIFARSubset(train_full, idx_task3, keep_classes=[4,5])

rng = random.Random(SEED)
k = min(NUM_FISHER_SAMPLES, len(train_task3))
sample_indices = rng.sample(range(len(train_task3)), k)
subset_task3 = Subset(train_task3, sample_indices)
subset_loader = DataLoader(subset_task3, batch_size=32, shuffle=False, num_workers=0)

# sanity check
check_labels = []
for i in range(len(subset_task3)):
    _, y = subset_task3[i]
    check_labels.append(int(y))
assert set(check_labels).issubset({0,1}), f"Found labels outside {{0,1}}: {set(check_labels)}"
print("[SANITY] Remapped labels OK → only {0,1}")

# ================== 4) Tools: ravel/unravel ==================
def unravel_index(index, shape):
    dims = []
    for s in reversed(shape):
        dims.append(index % s)
        index //= s
    return tuple(reversed(dims))

def ravel_index(multi_idx, shape):
    flat = 0
    for idx, dim in zip(multi_idx, shape):
        flat = flat * dim + idx
    return flat

# ================== 5) Fisher (EWC-Online style) ==================
def compute_fisher_per_sample(model, dataloader, device, head_name="task3"):
    model.train()
    fisher = {n: torch.zeros_like(p, device=device)
              for n, p in model.named_parameters() if p.requires_grad}

    total_samples = 0

    for inputs, targets in dataloader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        for ex, lab in zip(inputs, targets):
            ex  = ex.unsqueeze(0)
            lab = lab.unsqueeze(0)

            model.zero_grad(set_to_none=True)

            logits = model(ex, head_name)
            log_p  = logsoft(logits)                          # log p(y|x)
            loss_i = -F.nll_loss(log_p, lab, reduction='none')# = log p(y|x)
            p_i    = torch.exp(loss_i.detach())               # p(y|x)

            loss_scalar = loss_i.mean()
            loss_scalar.backward()

            w = float(p_i.mean().item())
            for name, p in model.named_parameters():
                if p.grad is not None:
                    fisher[name] += w * (p.grad.detach() ** 2)

            total_samples += 1

    for name in fisher:
        fisher[name] /= max(1, total_samples)

    return fisher

# ================== 6) Top-K & neighbors ==================
def get_topk_fisher_weights(fisher_dict, model_state_dict, k, exclude_prefixes=("heads.",)):
    if k <= 0:
        return []
    all_entries = []
    for name, f in fisher_dict.items():
        if any(name.startswith(pref) for pref in exclude_prefixes):
            continue
        if name not in model_state_dict:
            continue
        flat_f = f.flatten()
        flat_w = model_state_dict[name].flatten()
        for i in range(flat_f.numel()):
            all_entries.append({
                "name": name,
                "index": i,
                "value": float(flat_w[i].item()),
                "fisher": float(flat_f[i].item())
            })
    if not all_entries:
        return []
    all_entries.sort(key=lambda x: x["fisher"], reverse=True)
    k = min(k, len(all_entries))
    return all_entries[:k]

def extract_conv_neighbors(topk_entries, model, fisher_dict):
    neighbors = []
    if not topk_entries:
        return neighbors
    param_shapes = {name: p.shape for name, p in model.named_parameters()}
    fisher_flat = {name: tens.flatten() for name, tens in fisher_dict.items()}
    neighbor_offsets = [
        (0, 0, -1, -1), (0, 0, -1,  1),
        (0, 0,  1, -1), (0, 0,  1,  1),
        (0, 0, -1,  0), (0, 0,  1,  0),
        (0, 0,  0, -1), (0, 0,  0,  1),
    ]
    topk_set = set((e["name"], e["index"]) for e in topk_entries)
    added = set()
    for e in topk_entries:
        name, flat_idx = e["name"], e["index"]
        shape = param_shapes[name]
        if len(shape) != 4:
            continue  # Conv2d layers only
        oc, ic, kh, kw = unravel_index(flat_idx, shape)
        for do, di, dh, dw in neighbor_offsets:
            no, ni, nh, nw = oc + do, ic + di, kh + dh, kw + dw
            if 0 <= no < shape[0] and 0 <= ni < shape[1] and 0 <= nh < shape[2] and 0 <= nw < shape[3]:
                n_flat = ravel_index((no, ni, nh, nw), shape)
                key = (name, n_flat)
                if key in topk_set or key in added:
                    continue
                neighbors.append({
                    "name": name,
                    "index": n_flat,
                    "position": (no, ni, nh, nw),
                    "fisher": float(fisher_flat[name][n_flat].item())
                })
                added.add(key)
    return neighbors

# ================== 7) Build model, load checkpoint ==================
backbone = ResNet18Backbone(nf=64).to(DEVICE)
model = MultiHeadNet(backbone=backbone)
for t in ["task1","task2","task3","task4","task5"]:
    model.add_head(t, num_classes=2)
model.to(DEVICE)

ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
state_dict = ckpt["model_state"] if "model_state" in ckpt else ckpt
missing, unexpected = model.load_state_dict(state_dict, strict=False)
if missing:
    print("[INFO] Missing keys (not loaded):", missing)
if unexpected:
    print("[INFO] Unexpected keys (ignored):", unexpected)
model.train()

print("Loaded checkpoint:", MODEL_PATH)

# ================== 8) Fisher on CIFAR-10 Task-3 ==================
print("[INFO] Computing Fisher on CIFAR-10 (task3 classes 4&5, per-sample, diagonal)...")
fisher_info = compute_fisher_per_sample(model, subset_loader, DEVICE, head_name="task3")

with open(SAVE_FISHER_FULL, "wb") as f:
    pickle.dump({k: v.detach().cpu() for k, v in fisher_info.items()}, f)
print("Saved full Fisher to:", SAVE_FISHER_FULL)

topk_info = get_topk_fisher_weights(
    fisher_info,
    model.state_dict(),
    TOP_K,
    exclude_prefixes=("heads.",)
)
neighbors_info = extract_conv_neighbors(topk_info, model, fisher_info)

with open(SAVE_TOPK_PATH, "wb") as f:
    pickle.dump(topk_info, f)
with open(SAVE_NEIGHBORS_PATH, "wb") as f:
    pickle.dump(neighbors_info, f)

# ================== 9) Quick stats + AUDIT ==================
print(f"✅ Fisher computed from {len(subset_task3)} samples (per-sample, diagonal) on CIFAR-10 task3 (classes 4 & 5).")
print(f"✅ Top-K total (after excluding all heads.*): {len(topk_info)}")

# Audit: ensure Top-K contains no heads.* parameters
heads_in_topk = sum(1 for e in topk_info if e['name'].startswith("heads."))
print(f"[AUDIT] heads.* entries in Top-K: {heads_in_topk}  (Expected: 0)")

# Approximate distribution of Top-K elements across the backbone by parameter name
def cat_of_param(name):
    if ".conv" in name or "conv" in name:
        return "Conv"
    if ".gn" in name or "gn" in name.lower() or "norm" in name.lower():
        return "GN"
    return "Other"

from collections import Counter
cats = Counter(cat_of_param(e['name']) for e in topk_info)
print(f"[AUDIT] Top-K categories (backbone only): {dict(cats)}")

conv_topk = [e for e in topk_info if len(model.state_dict()[e['name']].shape) == 4] if topk_info else []
if topk_info:
    print(f"📌 Top-K from Conv2D: {len(conv_topk)} → {100*len(conv_topk)/len(topk_info):.2f}%")
else:
    print("📌 Top-K from Conv2D: 0")
print(f"📌 Neighbors extracted from Conv2D: {len(neighbors_info)}")

print("\n📊 Fisher Statistics per-parameter:")
print(f"{'Parameter':40s} | {'Mean':>12s} | {'Min':>12s} | {'Max':>12s}")
print("-"*85)
for name, tens in fisher_info.items():
    vals = tens.detach().cpu().view(-1)
    mean_val = vals.mean().item()
    min_val  = vals.min().item()
    max_val  = vals.max().item()
    print(f"{name:40s} | {mean_val:12.4e} | {min_val:12.4e} | {max_val:12.4e}")

print("✔️ Done.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[SANITY] Remapped labels OK → only {0,1}
Loaded checkpoint: /content/drive/MyDrive/ML_Project/project_files/First_benchmark/Gtask3_best_test_for_finetune.pth
[INFO] Computing Fisher on CIFAR-10 (task3 classes 4&5, per-sample, diagonal)...
Saved full Fisher to: /content/drive/MyDrive/ML_Project/project_files/First_benchmark/Gfisher_task3_cifar10_full.pkl
✅ Fisher computed from 2000 samples (per-sample, diagonal) on CIFAR-10 task3 (classes 4 & 5).
✅ Top-K total (after excluding all heads.*): 400000
[AUDIT] heads.* entries in Top-K: 0  (Expected: 0)
[AUDIT] Top-K categories (backbone only): {'Other': 19419, 'GN': 3055, 'Conv': 377526}
📌 Top-K from Conv2D: 396512 → 99.13%
📌 Neighbors extracted from Conv2D: 122617

📊 Fisher Statistics per-parameter:
Parameter                                |         Mean |          Min |          Max
------------------------------